# Set B 誤判：一鍵矩陣評估（Kaggle）

用途：跑 issue #67 的 S1 至 S3 —— 出 158 卡 × 6 條件矩陣、兩個 ONNX 模型推論、掃 `tau` 與拒判門檻，最後出一份 summary。

要點：

- 用 CPU 就夠，不需要 GPU 配額。
- 記得在 Notebook 設定開 **Internet**（要 clone public repo）。
- 不要把 19,000 張 PNG 存成 output（約 830 MB）；只保留 `summary.md`、`rows-*.csv`、`sweep-*.json`。
- `model-int8.onnx` 是 host 代理，並非 S31 上的 `.espdl`；正式數字仍要上板。

In [ ]:
!git clone --depth 1 https://github.com/Bruiseunderskin/smart-trash-can.git /kaggle/working/repo 2>/dev/null || true
%cd /kaggle/working/repo
!pip -q install numpy pillow onnxruntime

In [ ]:
!python3 tools/run_setb_matrix.py \
  --cards models/wordless-padded/cards \
  --cards-csv models/wordless-padded/cards.csv \
  --backgrounds models/wordless-padded/real-photos \
  --models models/wordless-padded/model-float32.onnx,models/wordless-padded/model-int8.onnx \
  --work /kaggle/working/setb \
  --per-condition 20 \
  --taus 0,0.5,1.0 \
  --min-confs 0.45,0.55,0.65,0.75 \
  --min-margins 0.05,0.15,0.25

In [ ]:
from pathlib import Path
print(Path('/kaggle/working/setb/summary.md').read_text())

## 讀結果

- `summary.md`：每個模型的掃描表，加每個模型的最佳操作點（覆蓋率 ≥ 0.85 優先）與 PASS/FAIL。
- `rows-<model>.csv`：逐張卡 × 條件 × 四類機率，可直接餵 `tools/stc_confusion.py`。
- `sweep-<model>.json`：完整掃描結果，含每類 P/R/F1、混淆矩陣、每卡準確率。

驗收線（issue #67）：印刷卡 nominal ≥ 99%、六條件平均 ≥ 95%、三張已知錯卡 ≥ 95%、自動化率 ≥ 85%、ECE ≤ 0.05。

如果 `tau` 校正後三張卡仍低於 95%，下一步是 S4 補樣本（藥盒、鋁箔包、外賣盒），而不是繼續調門檻。